# DEM Combine — model predictions -> georeferenced GeoTIFFs (.ipynb)

Runs the **per-patch `.npz` contract** end-to-end inside a notebook:

* Input: `tensors/train/{NNNNN}_{tile}_pXXXXX.npz` with keys
  `tensor (4,H,W)`, `geotransform (6,)`, `epsg`.
* Model sees **only Ch1** (FABDEM). Ch2/Ch3 are for the *other* pipeline and
  are ignored here. VDSR uses per-patch zero-centering (like your dataloader).
* Output: `out/{stem}_pred.tif` (float32), georeferenced **only** from the
  `.npz` `geotransform` + `epsg`. Optional `out/{stem}_diff.tif` vs `gt_visual`.

Set `CONFIG` below, then run the cells top-to-bottom. The last cell calls
`run_all(CONFIG)`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import torch

# model_vdsr.py lives here
NB_DIR = r"D:\Vaibhav\dem-lidar"
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

try:
    import rasterio
    from rasterio.transform import Affine
    from rasterio.crs import CRS
except ImportError:
    !pip install -q rasterio
    import rasterio
    from rasterio.transform import Affine
    from rasterio.crs import CRS

print("torch", torch.__version__, "| rasterio", rasterio.__version__)

torch 2.7.1+cu118 | rasterio 1.4.3


## 1. Configuration

Edit the paths / weights. `in_channels=[0]` + `norm="vdsr"` = Ch1-only VDSR
(matches the model you have). For the 3-channel net from the other pipeline you
would use `model_type="generic"`, `in_channels=[0,1,2]`, `norm="none"`.

In [2]:
CONFIG = {
    "tensors_dir":   r"D:\Vaibhav\dem-lidar\tensors_256\train",
    "gt_visual_dir": r"D:\Vaibhav\dem-lidar\gt_visual_256",
    "out_dir":       r"D:\Vaibhav\dem-lidar\out_vdsr",
    "weights":       r"D:\Vaibhav\dem-lidar\vdsr_checkpoints\checkpoint_epoch_0217.pth",
    "device":        "cuda",            # switch to "cpu" if no GPU
    "model_type":    "vdsr",
    "model_module":  "model_vdsr",
    "model_class":   "BaselineVDSR",
    "num_layers":    20,
    "num_features":  64,
    "in_channels":   [0],               # Ch1 only (Ch2/Ch3 ignored)
    "norm":          "vdsr",            # per-patch zero-centering
    "in_mean": None, "in_std": None,
    "out_mean": 0.0, "out_std": 1.0,
    "min_photons": None, "max_photons": None,
    "limit": None,                      # set e.g. 5 for a quick smoke test
    "write_diff": True,
}

## 2. Discover patches (filter by photon count)

In [3]:
import glob, re

def photon_count_of(stem):
    m = re.match(r"(\d+)_", stem)
    return int(m.group(1)) if m else -1

def discover_patches(cfg):
    files = sorted(glob.glob(str(Path(cfg["tensors_dir"]) / "*.npz")))
    sel = []
    for f in files:
        pc = photon_count_of(Path(f).stem)
        if cfg.get("min_photons") is not None and pc < cfg["min_photons"]:
            continue
        if cfg.get("max_photons") is not None and pc > cfg["max_photons"]:
            continue
        sel.append(f)
    if cfg.get("limit"):
        sel = sel[:cfg["limit"]]
    print(f"Selected {len(sel)} / {len(files)} .npz patches")
    return sel

## 3. Build the model

In [4]:
import sys
import torch

def build_model(cfg):
    # 1. Initialize the correct model architecture based on CONFIG
    if cfg["model_type"] == "vdsr":
        from model_vdsr import BaselineVDSR
        net = BaselineVDSR(num_layers=cfg["num_layers"], num_features=cfg["num_features"])
    elif cfg["model_type"] == "generic":
        mod = __import__(cfg["model_module"], fromlist=[cfg["model_class"]])
        net = getattr(mod, cfg["model_class"])()
    else:
        raise ValueError(cfg["model_type"])

    # 2. NumPy 2.0 -> 1.x Pickle Hack
    if 'numpy.core' in sys.modules:
        sys.modules['numpy._core'] = sys.modules['numpy.core']
        sys.modules['numpy._core.multiarray'] = sys.modules['numpy.core.multiarray']

    # 3. Load weights with security filter bypassed
    sd = torch.load(cfg["weights"], map_location="cpu", weights_only=False)
    
    # 4. Extract the actual weights (FIX APPLIED HERE)
    if isinstance(sd, dict):
        if "model_state_dict" in sd:
            sd = sd["model_state_dict"]
        elif "state_dict" in sd:
            sd = sd["state_dict"]
        
    # 5. Load weights strictly into the model
    net.load_state_dict(sd, strict=True)
    return net.eval().to(cfg["device"])

## 4. Predict one patch

Takes the selected channels, applies the chosen normalization, runs the model,
and returns a `(H, W)` float32 DEM. For `norm="vdsr"` it zero-centers Ch1 by its
patch mean and adds the mean back to the output.

In [5]:
def predict_patch(model, tensor, cfg):
    in_channels = cfg["in_channels"]
    in_ch = tensor[np.array(in_channels, dtype=int)].astype(np.float32)  # (Cin,H,W)
    dev = cfg["device"]
    norm = cfg["norm"]

    if norm == "vdsr":
        assert len(in_channels) == 1, "vdsr uses a single input channel (Ch1)"
        ch1 = in_ch[0]                          # (H, W)  <-- 2D, no channel dim
        nodata = cfg.get("nodata", -9999)
        valid = ch1 > nodata                    # (H, W)
        pm = float(ch1[valid].mean()) if valid.any() else 0.0

        xc = (ch1 - pm)[None, None, :, :]       # [1,1,H,W]
        xc[..., ~valid] = 0.0                   # mask NoData before the net
        with torch.no_grad():
            out = model(torch.from_numpy(xc).to(dev))
        out = np.squeeze(out.detach().cpu().numpy())
        if out.ndim == 3:
            out = out[0]

        # VDSR's skip already added the centered input back, so `out` is the
        # centered prediction. Restore ABSOLUTE elevation by adding the mean only.
        pred = (out + pm).astype(np.float32)   # <-- THE FIX (not out + in_ch)
        pred[~valid] = np.nan                  # restore NoData so QA/diff ignores it
        return pred

    if norm == "none":
        x = in_ch[None]                         # [1,Cin,H,W]
        with torch.no_grad():
            out = model(torch.from_numpy(x).to(dev))
        out = np.squeeze(out.detach().cpu().numpy())
        if out.ndim == 3:
            out = out[0]
        return out.astype(np.float32)

    if norm == "fixed":
        mean = np.array(cfg["in_mean"], dtype=np.float32)[:, None, None]
        std = np.array(cfg["in_std"], dtype=np.float32)[:, None, None]
        x = (in_ch - mean) / (std + 1e-8)
        x = x[None]
        with torch.no_grad():
            out = model(torch.from_numpy(x).to(dev))
        out = np.squeeze(out.detach().cpu().numpy())
        if out.ndim == 3:
            out = out[0]
        return (out * cfg["out_std"] + cfg["out_mean"]).astype(np.float32)

    raise ValueError(norm)

## 5. GeoTIFF export + run all patches

`write_geotiff` builds `Affine(dx, 0, x0, 0, -dy, y0)` from the `.npz`
geotransform and sets `CRS = EPSG(epsg)` — the model never sees these. Output
basename = `<stem>_pred.tif` (optionally `<stem>_diff.tif`).

In [6]:
def write_geotiff(pred, path, geotransform, epsg):
    gt = np.asarray(geotransform, dtype=np.float64)
    # GDAL (x0,dx,0, y0,0,-dy) -> rasterio Affine(a,b,c, d,e,f)
    transform = Affine(gt[1], gt[2], gt[0], gt[4], gt[5], gt[3])
    H, W = pred.shape
    with rasterio.open(path, "w", driver="GTiff", height=H, width=W,
                       count=1, dtype="float32", crs=CRS.from_epsg(int(epsg)),
                       transform=transform, compress="deflate", tiled=True) as dst:
        dst.write(pred.astype(np.float32), 1)
        dst.set_band_description(1, "elevation_m")

def run_all(cfg):
    out_dir = Path(cfg["out_dir"]); out_dir.mkdir(parents=True, exist_ok=True)
    gt_dir = Path(cfg["gt_visual_dir"]) if cfg.get("gt_visual_dir") else None
    model = build_model(cfg)
    npz_files = discover_patches(cfg)
    done = 0
    for npz_path in npz_files:
        stem = Path(npz_path).stem
        z = np.load(npz_path)
        tensor = z["tensor"].astype(np.float32)
        geotransform = z["geotransform"]
        epsg = int(z["epsg"])
        H, W = tensor.shape[1], tensor.shape[2]

        pred = predict_patch(model, tensor, cfg)            # (H, W) float32

        out_path = out_dir / f"{stem}_pred.tif"
        write_geotiff(pred, out_path, geotransform, epsg)
        done += 1

        if gt_dir is not None and cfg.get("write_diff", False):
            gt_path = gt_dir / f"{stem}.tif"
            if gt_path.exists():
                with rasterio.open(gt_path) as ds:
                    gt = ds.read(1).astype(np.float32)
                diff = np.abs(pred - gt)
                write_geotiff(diff, out_dir / f"{stem}_diff.tif", geotransform, epsg)
                valid = np.isfinite(gt) & np.isfinite(pred)
                if valid.any():
                    mae = float(np.abs(pred[valid] - gt[valid]).mean())
                    rmse = float(np.sqrt(((pred[valid] - gt[valid]) ** 2).mean()))
                    print(f"  {stem}: H={H} W={W} EPSG={epsg} MAE={mae:.3f} RMSE={rmse:.3f}")
        else:
            print(f"  {stem}: wrote {out_path.name} ({H}x{W}) EPSG={epsg}")
    print(f"Done. Wrote {done} prediction GeoTIFFs to {out_dir}")

## 6. Run

In [ ]:
# >>> RUN
if Path(CONFIG["tensors_dir"]).exists():
    run_all(CONFIG)
else:
    print("Edit CONFIG (tensors_dir, weights, ...) then run run_all(CONFIG)")

Selected 24 / 24 .npz patches
  00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00070: H=3000 W=1942 EPSG=32644 MAE=709.790 RMSE=2020.293
  00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00071: H=3000 W=1942 EPSG=32644 MAE=581.573 RMSE=1817.708
  00635_HMA_DEM8m_AT_20140225_0527_102001002B3B5E00_102001002B797800_DTM_10m_EGM08_p00032: H=3000 W=1961 EPSG=32644 MAE=123.857 RMSE=391.101
  00636_HMA_DEM8m_AT_20140225_0527_102001002B3B5E00_102001002B797800_DTM_10m_EGM08_p00029: H=3000 W=1961 EPSG=32644 MAE=129.251 RMSE=410.995
  00639_HMA_DEM8m_AT_20140225_0527_102001002B3B5E00_102001002B797800_DTM_10m_EGM08_p00028: H=3000 W=1961 EPSG=32644 MAE=131.981 RMSE=422.176
  00639_HMA_DEM8m_AT_20140225_0527_102001002B3B5E00_102001002B797800_DTM_10m_EGM08_p00030: H=3000 W=1961 EPSG=32644 MAE=127.279 RMSE=403.565
  00640_HMA_DEM8m_AT_20140225_0527_102001002B3B5E00_102001002B797800_DTM_10m_EGM08_p00027: H=3000 W=1961 EPSG=32644 MAE=13

## 7. Verify an output

Confirms the prediction kept its own georeferencing (CRS + transform + shape).

In [ ]:
# >>> VERIFY
import glob as _glob
outs = sorted(_glob.glob(str(Path(CONFIG["out_dir"]) / "*_pred.tif")))
if outs:
    with rasterio.open(outs[0]) as ds:
        print(Path(outs[0]).name, "shape", ds.shape, "crs", ds.crs)
        print("transform", [round(v, 3) for v in ds.transform][:6])
else:
    print("No outputs yet - run run_all(CONFIG) first.")